# 🧁 Day 1: Bakery Shop AI Agent
### Powered by LangChain (`ChatGroq`), Groq (`openai/gpt-oss-120b`), & Streamlit

This agent serves **Sweet Crumbs Bakery & Cafe**. It is built using **LangChain Groq**, governed by **exact facts and operational rules**, and connected to a structured menu database.

In [1]:
# Install required LangChain and Groq dependencies
!pip install langchain langchain-groq langchain-core groq streamlit python-dotenv -q

In [2]:
import os
from dotenv import load_dotenv

# Load API key securely from .env file or environment
load_dotenv()

# Try retrieving from Colab Secrets if running in Google Colab
try:
    from google.colab import userdata
    secret_key = userdata.get('GROQ_API_KEY')
    if secret_key:
        os.environ['GROQ_API_KEY'] = secret_key
except Exception:
    pass

if not os.getenv('GROQ_API_KEY'):
    print('Warning: GROQ_API_KEY not found. Please set it in your .env file or Colab secrets.')


In [3]:
# Initialize the LangChain ChatGroq model: openai/gpt-oss-120b
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.1,
    groq_api_key=os.environ["GROQ_API_KEY"]
)
print("ChatGroq initialized successfully with model: openai/gpt-oss-120b")

## 📋 1. Bakery Menu Database & Categories
Structured database containing menu categories, items, pricing, dietary flags, and descriptions.

In [4]:
# Bakery Database
BAKERY_MENU = {
    "Cakes": [
        {"name": "Belgian Dark Chocolate Cake", "price": "$28 (500g) / $50 (1kg)", "dietary": "Egg & Eggless available", "desc": "Rich 70% dark Belgian ganache with moist sponge."},
        {"name": "Red Velvet Cream Cheese Cake", "price": "$30 (500g) / $55 (1kg)", "dietary": "Egg & Eggless available", "desc": "Traditional cocoa sponge layered with vanilla bean cream cheese frosting."},
        {"name": "Classic Vanilla Bean Cake", "price": "$24 (500g) / $45 (1kg)", "dietary": "100% Eggless", "desc": "Light sponge infused with pure Madagascar vanilla."},
        {"name": "Fresh Mango Mascarpone (Seasonal)", "price": "$32 (500g) / $58 (1kg)", "dietary": "Eggless available", "desc": "Fresh Alphonso mango slices with whipped mascarpone."}
    ],
    "Pastries & Desserts": [
        {"name": "Blueberry Baked Cheesecake Slice", "price": "$6.00", "dietary": "Contains Egg", "desc": "New York style rich cheesecake with blueberry compote."},
        {"name": "Dutch Truffle Pastry", "price": "$5.00", "dietary": "100% Eggless", "desc": "Layered chocolate truffle pastry slice."},
        {"name": "Choco Lava Molten Cupcake", "price": "$4.50", "dietary": "Eggless available", "desc": "Warm gooey molten chocolate center."},
        {"name": "Assorted French Macarons (Box of 6)", "price": "$14.00", "dietary": "Contains Egg (Gluten-Free)", "desc": "Pistachio, Raspberry, Salted Caramel, Chocolate, Vanilla, Lemon."}
    ],
    "Artisanal Breads": [
        {"name": "Classic Country Sourdough Loaf", "price": "$8.00", "dietary": "Vegan / 100% Eggless", "desc": "36-hour naturally fermented sourdough with crisp crust."},
        {"name": "Garlic Herb Rosemary Focaccia", "price": "$7.00", "dietary": "Vegetarian", "desc": "Extra virgin olive oil dough topped with roasted garlic and fresh herbs."},
        {"name": "Butter Croissant", "price": "$4.00", "dietary": "Contains Egg Wash & Dairy", "desc": "Flaky 27-layer French butter croissant."},
        {"name": "Seeded Multigrain Sandwich Bread", "price": "$6.50", "dietary": "Vegan / 100% Eggless", "desc": "Wholesome bread baked with flax, pumpkin, and sunflower seeds."}
    ],
    "Cookies & Savory": [
        {"name": "Double Choc-Chip Walnut Cookie", "price": "$3.50", "dietary": "Eggless available", "desc": "Crispy edges with a chewy chocolate center."},
        {"name": "Almond Pistachio Biscotti (Pack of 4)", "price": "$7.00", "dietary": "100% Eggless", "desc": "Twice-baked crunchy Italian biscuits, perfect with coffee."},
        {"name": "Paneer Spiced Puff Pastry", "price": "$4.50", "dietary": "Vegetarian", "desc": "Flaky golden crust stuffed with spiced cottage cheese filling."}
    ]
}

BAKERY_INFO = {
    "name": "Sweet Crumbs Bakery & Cafe",
    "hours": "Tuesday to Sunday, 9:00 AM – 8:00 PM (Closed on Mondays)",
    "phone": "+1-555-019-2253",
    "location": "12 Baker Street, Gourmet District",
    "delivery": "Local delivery within 10 km via courier partners; in-store pickup is always free."
}

def format_menu():
    lines = []
    for cat, items in BAKERY_MENU.items():
        lines.append(f"[{cat}]")
        for i in items:
            lines.append(f"- {i['name']} | Price: {i['price']} | Diet: {i['dietary']} | Notes: {i['desc']}")
    return "\n".join(lines)

## 📜 2. Exact Bakery Facts & Rules Prompt
Specific operational facts and strict rules governing the bakery agent.

In [5]:
SYSTEM = f"""You are the assistant for {BAKERY_INFO['name']}.

=== BAKERY FACTS ===
- Hours: {BAKERY_INFO['hours']}
- Location: {BAKERY_INFO['location']}
- Contact: {BAKERY_INFO['phone']}
- Notice period: Standard cakes require minimum 24 hours' notice. Custom or tiered cakes require 48 hours' notice.
- In-store counter items: Individual pastries, sourdough, and croissants are freshly baked daily and available while supplies last.
- Diet & Allergies: ALL cakes can be baked in 100% eggless versions upon request. Sourdough and Multigrain breads are 100% vegan.
- Pickup & Delivery: Store pickup is free. Delivery is available within 10 km via courier services.

=== MENU DATABASE ===
{format_menu()}

=== STRICT RULES ===
1. NEVER promise or guarantee exact delivery times (always state delivery arrival depends on courier schedules and traffic).
2. NEVER accept custom cake designs or custom pricing directly; always direct customers to call or WhatsApp {BAKERY_INFO['phone']}.
3. If an item or detail is unknown or not on the menu, politely say you do not know and invite them to call the bakery.
4. Format: Reply warmly, politely, and concisely (under 70 words).
"""

## 💬 3. LangChain Invocation Function

In [6]:
def ask(question: str) -> str:
    """Invokes the LangChain ChatGroq model with System facts/rules and Human question."""
    messages = [
        SystemMessage(content=SYSTEM),
        HumanMessage(content=question)
    ]
    response = llm.invoke(messages)
    return response.content

## 🧪 4. Testing Bakery Scenarios

In [7]:
test_questions = [
    "Can I get my order delivered in 2 hours today?",
    "Do you have eggless cake options available?",
    "What sourdough breads do you have and what are their prices?"
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}\n")

Q: Can I get my order delivered in 2 hours today?
A: We’d love to get treats to you quickly! However, our cakes require at least 24 hours' notice, and we can't guarantee exact delivery times as they depend on courier traffic. Feel free to call us at +1-555-019-2253 to check today's counter availability!

Q: Do you have eggless cake options available?
A: Yes! Every cake on our menu has a 100% eggless version available—including Belgian Dark Chocolate, Red Velvet Cream Cheese, Classic Vanilla Bean, and Fresh Mango Mascarpone.

Q: What sourdough breads do you have and what are their prices?
A: We bake our Classic Country Sourdough Loaf ($8.00, 100% vegan) and Garlic Herb Rosemary Focaccia ($7.00). Both are baked fresh daily and available in-store while supplies last!


## 🚀 5. Interactive Streamlit Web App
To launch the full interactive web application:
```bash
streamlit run app.py
```